In [ ]:
import numpy as np
import os
import sys, os; sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), '..')))
from utils.model_loader import get_model_fits
import numpy as np
import pandas as pd
import re
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from utils.generate_data import load_linreg_dataset, generate_linreg_simple_data

#rhos = [-0.9, -0.5, 0.0, 0.5, 0.9]
rhos = [0.0, 0.5, 0.9]

# Map rho to the correct results directory
results_dir_map = {
    #-0.9: "results/regression/linreg/high_neg_corr",
    #-0.5: "results/regression/linreg/medium_neg_corr",
    0.0: "results/regression/linreg/no_corr",
    0.5: "results/regression/linreg/medium_corr",
    0.9: "results/regression/linreg/high_corr",
}

model_names = [
    "Linreg Gaussian",
    "Linreg Regularized Horseshoe scaled",
    "Linreg Dirichlet Horseshoe",
    "Linreg Dirichlet Student T",
    "Linreg Beta Horseshoe",
    "Linreg Beta Student T",
]

data_dir = "datasets/linreg"

# This will hold everything in an easy-to-handle structure
experiments = {}

for rho in rhos:
    # 1. Load data
    dataset_path = f"{data_dir}/linreg_data_rho_{rho}.npz"
    X_train, X_test, y_train, y_test, _, _, _ = load_linreg_dataset(
        path=dataset_path,
        test_fraction=0.2,
        seed=123,
    )

    # 2. True coefficients
    _, _, beta_true = generate_linreg_simple_data(rho=rho)

    # 3. Model fits
    results_dir_linreg = results_dir_map[rho]
    full_config_path = f"linreg_N200_p10_rho_{rho}"

    linreg_fit = get_model_fits(
        config=full_config_path,
        results_dir=results_dir_linreg,
        models=model_names,
        include_prior=False,
    )

    # 4. Store everything nicely under this rho
    experiments[rho] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "beta_true": beta_true,
        "results_dir": results_dir_linreg,
        "config": full_config_path,
        "fits": linreg_fit,
    }


In [ ]:
import numpy as np

rmse_results = {}

for rho, exp in experiments.items():
    X_train = exp["X_train"]
    X_test  = exp["X_test"]
    y_train = exp["y_train"]
    y_test  = exp["y_test"]
    fits    = exp["fits"]

    # Posterior samples
    beta_gauss = fits['Linreg Gaussian']['posterior'].stan_variable("beta")
    beta_RHS   = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("beta")
    beta_DHS   = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("beta")
    beta_DST   = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("beta")
    beta_BHS   = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("beta")
    beta_BST   = fits['Linreg Beta Student T']['posterior'].stan_variable("beta")

    # GLS / OLS baseline
    beta_GLS = np.linalg.pinv(X_train.T @ X_train) @ (X_train.T @ y_train)

    # RMSEs based on posterior means
    rmse_gauss = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_gauss, axis=0))**2))
    rmse_RHS   = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_RHS,   axis=0))**2))
    rmse_DHS   = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_DHS,   axis=0))**2))
    rmse_DST   = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_DST,   axis=0))**2))
    rmse_BHS   = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_BHS,   axis=0))**2))
    rmse_BST   = np.sqrt(np.mean((y_test - X_test @ np.mean(beta_BST,   axis=0))**2))
    rmse_GLS   = np.sqrt(np.mean((y_test - X_test @ beta_GLS)**2))

    rmse_results[rho] = {
        "Gaussian"       : rmse_gauss,
        "Regularized HS" : rmse_RHS,
        "Dirichlet HS"   : rmse_DHS,
        "Dirichlet ST"   : rmse_DST,
        "Beta HS"        : rmse_BHS,
        "Beta ST"        : rmse_BST,
        "GLS"            : rmse_GLS,
    }

# Nice printout
for rho in sorted(rmse_results.keys()):
    res = rmse_results[rho]
    print(f"\nRMSE summary for rho = {rho}")
    print("-" * 35)
    print(f"Gaussian        : {res['Gaussian']:.4f}")
    print(f"Regularized HS  : {res['Regularized HS']:.4f}")
    print(f"Dirichlet HS    : {res['Dirichlet HS']:.4f}")
    print(f"Dirichlet ST    : {res['Dirichlet ST']:.4f}")
    print(f"Beta HS         : {res['Beta HS']:.4f}")
    print(f"Beta ST         : {res['Beta ST']:.4f}")
    print(f"GLS             : {res['GLS']:.4f}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- choose which rho to plot ---
rho_to_plot = 0.9   # change to 0.0 or 0.5 if you like

exp = experiments[rho_to_plot]
fits = exp["fits"]
beta_true = exp["beta_true"]

# Extract posterior draws for this rho
beta_gauss = fits['Linreg Gaussian']['posterior'].stan_variable("beta")
beta_RHS   = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("beta")
beta_DHS   = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("beta")
beta_DST   = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("beta")
beta_BHS   = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("beta")
beta_BST   = fits['Linreg Beta Student T']['posterior'].stan_variable("beta")

S, P = beta_gauss.shape  # number of draws, number of coefficients

# Put all draws into one long DataFrame
def beta_to_long_df(beta_array, model_name):
    """
    beta_array: (S, P)
    returns DataFrame with columns: model, draw, coeff, beta
    """
    S, P = beta_array.shape
    df = pd.DataFrame(
        beta_array.reshape(S * P),
        columns=["beta"]
    )
    df["draw"] = np.repeat(np.arange(S), P)
    df["coeff"] = np.tile(np.arange(P), S)
    df["model"] = model_name
    return df

df_gauss = beta_to_long_df(beta_gauss, "Gaussian")
df_RHS   = beta_to_long_df(beta_RHS,   "Regularized Horseshoe")
df_DHS   = beta_to_long_df(beta_DHS,   "Dirichlet Horseshoe")
df_DST   = beta_to_long_df(beta_DST,   "Dirichlet Student T")
df_BHS   = beta_to_long_df(beta_DHS,   "Beta Horseshoe")
df_BST   = beta_to_long_df(beta_DST,   "Beta Student T")

#beta_df = pd.concat([df_gauss, df_RHS, df_DHS, df_DST], ignore_index=True)
beta_df = pd.concat([df_gauss, df_RHS, df_DHS, df_DST, df_BHS, df_BST], ignore_index=True)

# Attach true beta from experiments
beta_true_series = pd.Series(beta_true, index=np.arange(len(beta_true)))
beta_df["beta_true"] = beta_df["coeff"].map(beta_true_series)

# Boxplot per coefficient, grouped by model
coeffs_to_plot = 6
rows = 3
cols = int(np.ceil(coeffs_to_plot / rows))

fig, axes = plt.subplots(rows, cols, figsize=(6, 6), sharey=False)
axes = axes.flatten()

for j in range(coeffs_to_plot):
    ax = axes[j]
    df_j = beta_df[beta_df["coeff"] == j]

    # Boxplot of posterior for beta_j under each model
    data = [
        df_j[df_j["model"] == m]["beta"].values
        for m in ["Gaussian", "Regularized Horseshoe", "Dirichlet Horseshoe", "Dirichlet Student T", "Beta Horseshoe", "Beta Student T"]
    ]
    ax.boxplot(data, showfliers=False)
    #ax.set_xticks([1, 2, 3, 4])
    ax.set_xticks([1, 2, 3, 4, 5, 6])
    #ax.set_xticklabels(["Gauss", "RHS", "DHS", "DST"], rotation=30)
    ax.set_xticklabels(["Gauss", "RHS", "DHS", "DST", "BHS", "BST"], rotation=30)
    ax.set_title(fr"$w_{{{j+1}}}$")

    # True beta as horizontal line
    ax.axhline(beta_true_series[j], linestyle="--", linewidth=1)

# Hide unused axes if any
for k in range(P, len(axes)):
    axes[k].axis("off")

fig.suptitle(fr"Posterior distributions of $w_j$ by prior, $\rho={rho}$", fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- choose which rho to work with ---
rho_to_plot = 0.9  # or 0.0, 0.5

exp = experiments[rho_to_plot]
fits = exp["fits"]
X_train = exp["X_train"]
beta_true = exp["beta_true"]

# --- extract posterior draws for global/local scales and betas ---
beta_gauss = fits['Linreg Gaussian']['posterior'].stan_variable("beta")
sigma_gauss = fits['Linreg Gaussian']['posterior'].stan_variable("sigma")

beta_RHS = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("beta")
sigma_RHS = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("sigma")
tau_RHS   = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("tau")
lambda_RHS = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("lambda_tilde")

beta_DHS = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("beta")
sigma_DHS = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("sigma")
tau_DHS   = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("tau")
lambda_DHS = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("lambda_data")
xi_DHS     = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("phi_data")

beta_DST = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("beta")
sigma_DST = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("sigma")
tau_DST   = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("tau")
lambda_DST = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("lambda_tilde")
xi_DST     = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("phi_data")

beta_BHS = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("beta")
sigma_BHS = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("sigma")
tau_BHS   = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("tau")
lambda_BHS = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("lambda_data")
xi_BHS     = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("phi_data")

beta_BST = fits['Linreg Beta Student T']['posterior'].stan_variable("beta")
sigma_BST = fits['Linreg Beta Student T']['posterior'].stan_variable("sigma")
tau_BST   = fits['Linreg Beta Student T']['posterior'].stan_variable("tau")
lambda_BST = fits['Linreg Beta Student T']['posterior'].stan_variable("lambda_tilde")
xi_BST     = fits['Linreg Beta Student T']['posterior'].stan_variable("phi_data")

# GLS baseline
beta_GLS = np.linalg.pinv(X_train.T @ X_train) @ (X_train.T @ exp["y_train"])

# infer dimensions
S, p = beta_RHS.shape       # number of posterior draws, number of covariates
N = X_train.shape[0]        # sample size used in the model

# --- compute shrinkage factors kappa and effective parameters m_eff ---
kappa_gauss = np.zeros((S, p))
kappa_RHS = np.zeros((S, p))
kappa_DHS = np.zeros((S, p))
kappa_DST = np.zeros((S, p))
kappa_BHS = np.zeros((S, p))
kappa_BST = np.zeros((S, p))

meff_gauss = np.zeros(S)
meff_RHS = np.zeros(S)
meff_DHS = np.zeros(S)
meff_DST = np.zeros(S)
meff_BHS = np.zeros(S)
meff_BST = np.zeros(S)

for i in range(S):
     # RHS: kappa_j = 1 / (1 + N * sigma^-2 * tau^2 * lambda_j)
    kappa_gauss[i] = 1.0 / (1.0 + N * sigma_gauss[i]**(-2))
    
    # RHS: kappa_j = 1 / (1 + N * sigma^-2 * tau^2 * lambda_j)
    kappa_RHS[i] = 1.0 / (1.0 + N * sigma_RHS[i]**(-2) * tau_RHS[i]**2 * lambda_RHS[i])

    # DHS: kappa_j = 1 / (1 + N * sigma^-2 * tau^2 * lambda_j * xi_j)
    kappa_DHS[i] = 1.0 / (1.0 + N * sigma_DHS[i]**(-2) * tau_DHS[i]**2 * lambda_DHS[i] * xi_DHS[i])

    # DST: same structure, but with DST parameters (note: tau_DST, not tau_DHS)
    kappa_DST[i] = 1.0 / (1.0 + N * sigma_DST[i]**(-2) * tau_DST[i]**2 * lambda_DST[i] * xi_DST[i])
    
    # BHS: kappa_j = 1 / (1 + N * sigma^-2 * tau^2 * lambda_j * xi_j)
    kappa_BHS[i] = 1.0 / (1.0 + N * sigma_BHS[i]**(-2) * tau_BHS[i]**2 * lambda_BHS[i] * xi_BHS[i])

    # BST: same structure, but with DST parameters (note: tau_DST, not tau_DHS)
    kappa_BST[i] = 1.0 / (1.0 + N * sigma_BST[i]**(-2) * tau_BST[i]**2 * lambda_BST[i] * xi_BST[i])

    meff_gauss[i] = np.sum(1.0 - kappa_gauss[i])
    meff_RHS[i] = np.sum(1.0 - kappa_RHS[i])
    meff_DHS[i] = np.sum(1.0 - kappa_DHS[i])
    meff_DST[i] = np.sum(1.0 - kappa_DST[i])
    meff_BHS[i] = np.sum(1.0 - kappa_BHS[i])
    meff_BST[i] = np.sum(1.0 - kappa_BST[i])

In [ ]:
from scipy.stats import gaussian_kde

# Indices of coefficients to visualize
idxs = [0, 4, 5]

# If you want to hard-code "true" betas, use this:
beta_true_vals = {0: 3, 4: 0.2, 5: 0.0}
titles = {
    0: r"$\beta_1$", #1: r"$\beta_2$", 
    #2: r"$\beta_3$", 3: r"$\beta_4$", 
    4: r"$\beta_5$", 5: r"$\beta_6$"
}

def common_bins(*arrays, bins=40, range=None):
    """Compute common histogram bin edges for multiple arrays."""
    data = np.concatenate([a.ravel() for a in arrays])
    return np.histogram_bin_edges(data, bins=bins, range=range)

fig, axes = plt.subplots(len(idxs), 2, figsize=(10, 8), sharex=False, sharey=False)

for row, j in enumerate(idxs):
    ax_kappa = axes[row, 1]
    ax_beta  = axes[row, 0]

    # --- Kappa posterior ---
    bins_kappa = common_bins(kappa_gauss[:, j], kappa_RHS[:, j], kappa_DHS[:, j], kappa_DST[:, j],
                              bins=40, range=(0, 1.0))
    ax_kappa.hist(kappa_gauss[:, j], bins=bins_kappa, alpha=0.6, label="Gauss", density=True, color = "C0", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_RHS[:, j], bins=bins_kappa, alpha=0.6, label="RHS", density=True, color = "C1", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_DHS[:, j], bins=bins_kappa, alpha=0.6, label="DHS", density=True, color = "C2", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_DST[:, j], bins=bins_kappa, alpha=0.6, label="DST", density=True, color = "C3", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_BHS[:, j], bins=bins_kappa, alpha=0.6, label="BHS", density=True, color = "C4", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_BST[:, j], bins=bins_kappa, alpha=0.6, label="BST", density=True, color = "C5", histtype="step", linewidth=2)
    ax_kappa.set_xlabel(fr"$\kappa_{j+1}$", fontsize=15)


    # --- Beta posterior ---
    bins_beta = common_bins(beta_gauss[:, j], beta_RHS[:, j], beta_DHS[:, j], beta_DST[:, j], bins=40)
    x = np.linspace(bins_beta[0], bins_beta[-1], 500)
    ax_beta.plot(x, gaussian_kde(beta_gauss[:, j], bw_method=0.25)(x), label="Gauss")
    ax_beta.plot(x, gaussian_kde(beta_RHS[:, j], bw_method=0.25)(x), label="RHS", color="C1")
    ax_beta.plot(x, gaussian_kde(beta_DHS[:, j], bw_method=0.25)(x), label="DHS", color="C2")
    ax_beta.plot(x, gaussian_kde(beta_DST[:, j], bw_method=0.25)(x), label="DST", color="C3")
    ax_beta.plot(x, gaussian_kde(beta_BHS[:, j], bw_method=0.25)(x), label="BHS", color="C4")
    ax_beta.plot(x, gaussian_kde(beta_BST[:, j], bw_method=0.25)(x), label="BST", color="C5")


    ax_beta.axvline(beta_true_vals[j], alpha=0.9, label="w true", color="black", linestyle="--")
    ax_beta.axvline(beta_GLS[j], alpha=0.9, label="w GLS", color="purple", linestyle=":")
    ax_beta.set_xlabel(fr"$w_{j+1}$", fontsize=15)
    ax_beta.set_ylabel("Density", fontsize=15)
    #ax_beta.set_title(f"Beta, {titles[j]}")

# One legend for beta panels
handles, labels = axes[0, 0].get_legend_handles_labels()

fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.45, 1))
fig.tight_layout(rect=[0, 0, 0.85, 0.93])
fig.savefig("figures_for_use_in_paper/w_vs_kappa.pdf", bbox_inches="tight")
plt.show()


In [ ]:

# ------------------------------------------------------
# 2) Plot posterior distribution of effective parameters
# ------------------------------------------------------

P = p
bins = np.linspace(2, 10, 30)#np.arange(0, P + 2) - 0.1  # bin edges at integers

plt.figure(figsize=(6, 5))

# Use plt.hist to keep dependencies minimal
plt.hist(meff_gauss, bins=bins, density=True, alpha=0.6, label="Gauss", histtype="stepfilled")
plt.hist(meff_RHS, bins=bins, density=True, alpha=0.6, label="RHS", histtype="stepfilled")
plt.hist(meff_DHS, bins=bins, density=True, alpha=0.6, label="DHS", histtype="stepfilled")
plt.hist(meff_DST, bins=bins, density=True, alpha=0.6, label="DST", histtype="stepfilled")
plt.hist(meff_BHS, bins=bins, density=True, alpha=0.6, label="BHS", histtype="stepfilled")
plt.hist(meff_BST, bins=bins, density=True, alpha=0.6, label="BST", histtype="stepfilled")

# vertical line at true number of active coefficients
true_active = 4  # adjust if needed
plt.axvline(true_active, color="black", linestyle="--", linewidth=1.5,
            label=f"True active = {true_active}")

# add posterior means as vertical lines + text
ymax = plt.ylim()[1]
for Meff, label, color in [
    (meff_gauss, "Gauss", "C0"),
    (meff_RHS, "RHS", "C1"),
    (meff_DHS, "DHS", "C2"),
    (meff_DST, "DST", "C3"),
    (meff_BHS, "BHS", "C4"),
    (meff_BST, "BST", "C5"),
]:
    mean_val = np.mean(Meff)
    plt.axvline(mean_val, color=color, linestyle=":", linewidth=1.5)
    plt.text(mean_val + 0.1, ymax * 0.8,
             f"{label} mean={mean_val:.1f}",
             color="black", fontsize=9, rotation=90, va="top")

plt.xticks(range(0, P + 1))
plt.xlabel(r"Effective number of parameters $m_{\mathrm{eff}}$")
plt.ylabel("Posterior density")
plt.title(fr"Posterior samples of effective parameters, $\rho={rho_to_plot}$")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

model_names = [
    "Linreg Gaussian",
    "Linreg Regularized Horseshoe scaled",
    "Linreg Dirichlet Horseshoe",
    "Linreg Dirichlet Student T",
    "Linreg Beta Horseshoe",
    "Linreg Beta Student T",
]

def beta_to_long_df(beta_array, model_name, rho, beta_true):
    """
    beta_array: (S, P)
    returns DataFrame with columns: rho, model, draw, coeff, beta, beta_true
    """
    S, P = beta_array.shape
    df = pd.DataFrame(
        beta_array.reshape(S * P),
        columns=["beta"]
    )
    df["draw"] = np.repeat(np.arange(S), P)
    df["coeff"] = np.tile(np.arange(P), S)
    df["model"] = model_name
    df["rho"] = rho
    # attach true beta per coefficient
    beta_true_series = pd.Series(beta_true, index=np.arange(len(beta_true)))
    df["beta_true"] = df["coeff"].map(beta_true_series)
    return df

all_dfs = []

for rho, exp in experiments.items():
    fits = exp["fits"]
    beta_true = exp["beta_true"]

    beta_gauss = fits['Linreg Gaussian']['posterior'].stan_variable("beta")
    beta_RHS   = fits['Linreg Regularized Horseshoe scaled']['posterior'].stan_variable("beta")
    beta_DHS   = fits['Linreg Dirichlet Horseshoe']['posterior'].stan_variable("beta")
    beta_DST   = fits['Linreg Dirichlet Student T']['posterior'].stan_variable("beta")
    beta_BHS   = fits['Linreg Beta Horseshoe']['posterior'].stan_variable("beta")
    beta_BST   = fits['Linreg Beta Student T']['posterior'].stan_variable("beta")

    all_dfs.append(beta_to_long_df(beta_gauss, "Gauss",             rho, beta_true))
    all_dfs.append(beta_to_long_df(beta_RHS,   "RHS",       rho, beta_true))
    all_dfs.append(beta_to_long_df(beta_DHS,   "DHS",         rho, beta_true))
    all_dfs.append(beta_to_long_df(beta_DST,   "DST",  rho, beta_true))
    all_dfs.append(beta_to_long_df(beta_BHS,   "BHS",         rho, beta_true))
    all_dfs.append(beta_to_long_df(beta_BST,   "BST",  rho, beta_true))

beta_all = pd.concat(all_dfs, ignore_index=True)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# coefficients to visualize
coeffs_to_plot = [0, 4, 5]#[0, 1, 2, 3, 4, 5]   # β1,…,β6
rows, cols = 1, 3

fig, axes = plt.subplots(rows, cols, figsize=(10, 5), sharey=False)
axes = axes.flatten()

for idx, j in enumerate(coeffs_to_plot):
    ax = axes[idx]
    df_j = beta_all[beta_all["coeff"] == j].copy()

    # nice ordering for models and rho
    df_j["model"] = pd.Categorical(
        df_j["model"],
        categories=["Gauss", "RHS", "DHS", "DST"], #, "BHS", "BST"],
        ordered=True,
    )
    df_j["rho"] = df_j["rho"].astype(float)
    

    sns.boxplot(
        data=df_j,
        x="model",
        y="beta",
        hue="rho",
        ax=ax,
        showfliers=False,
    )

    # true beta line
    beta_true_j = df_j["beta_true"].iloc[0]
    ax.axhline(beta_true_j, linestyle="--", linewidth=1)

    ax.set_xlabel("")
    ax.set_ylabel(fr"$w_{{{j+1}}}$", fontsize=15, rotation=0, labelpad=15)
    #ax.set_title(fr"$w_{{{j+1}}}$", fontsize=11)
    ax.tick_params(axis="x", rotation=20, labelsize=13)

# remove extra axes if any
for k in range(len(coeffs_to_plot), len(axes)):
    axes[k].axis("off")

# one legend for the whole figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title=r"$\rho$", loc="upper right", bbox_to_anchor=(0.97, 0.9), ncol=1, fontsize=12)
for ax in axes:
    ax.get_legend().remove()

#fig.suptitle(r"Posterior distributions of $w_j$ by prior and correlation $\rho$", fontsize=13)
fig.tight_layout(rect=[0, 0.3, 1, 0.95])
fig.savefig("figures_for_use_in_paper/w_distribution.pdf", bbox_inches="tight")
plt.show()


In [ ]:
from scipy.stats import gaussian_kde

# Indices of coefficients to visualize
idxs = [0, 4, 5]

# If you want to hard-code "true" betas, use this:
beta_true_vals = {0: 3, 4: 0.2, 5: 0.0}
titles = {
    0: r"$\beta_1$", #1: r"$\beta_2$", 
    #2: r"$\beta_3$", 3: r"$\beta_4$", 
    4: r"$\beta_5$", 5: r"$\beta_6$"
}

def common_bins(*arrays, bins=40, range=None):
    """Compute common histogram bin edges for multiple arrays."""
    data = np.concatenate([a.ravel() for a in arrays])
    return np.histogram_bin_edges(data, bins=bins, range=range)

fig, axes = plt.subplots(2, len(idxs), figsize=(10, 5), sharex=False, sharey=False)

for row, j in enumerate(idxs):
    ax_kappa = axes[0, row]
    ax_beta  = axes[1, row]

    # --- Kappa posterior ---
    bins_kappa = common_bins(kappa_gauss[:, j], kappa_RHS[:, j], kappa_DHS[:, j], kappa_DST[:, j],
                              bins=40, range=(0, 1.0))
    ax_kappa.hist(kappa_gauss[:, j], bins=bins_kappa, alpha=0.6, label="Gauss", density=True, color = "C0", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_RHS[:, j], bins=bins_kappa, alpha=0.6, label="RHS", density=True, color = "C1", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_DHS[:, j], bins=bins_kappa, alpha=0.6, label="DHS", density=True, color = "C2", histtype="step", linewidth=2)
    ax_kappa.hist(kappa_DST[:, j], bins=bins_kappa, alpha=0.6, label="DST", density=True, color = "C3", histtype="step", linewidth=2)
    # ax_kappa.hist(kappa_BHS[:, j], bins=bins_kappa, alpha=0.6, label="BHS", density=True, color = "C4", histtype="step", linewidth=2)
    # ax_kappa.hist(kappa_BST[:, j], bins=bins_kappa, alpha=0.6, label="BST", density=True, color = "C5", histtype="step", linewidth=2)
    ax_kappa.set_xlabel(fr"$\kappa_{j+1}$", fontsize=15)


    # --- Beta posterior ---
    bins_beta = common_bins(beta_gauss[:, j], beta_RHS[:, j], beta_DHS[:, j], beta_DST[:, j], bins=40)
    x = np.linspace(bins_beta[0], bins_beta[-1], 500)
    ax_beta.plot(x, gaussian_kde(beta_gauss[:, j], bw_method=0.25)(x), label="Gauss")
    ax_beta.plot(x, gaussian_kde(beta_RHS[:, j], bw_method=0.25)(x), label="RHS", color="C1")
    ax_beta.plot(x, gaussian_kde(beta_DHS[:, j], bw_method=0.25)(x), label="DHS", color="C2")
    ax_beta.plot(x, gaussian_kde(beta_DST[:, j], bw_method=0.25)(x), label="DST", color="C3")
    # ax_beta.plot(x, gaussian_kde(beta_BHS[:, j], bw_method=0.25)(x), label="BHS", color="C4")
    # ax_beta.plot(x, gaussian_kde(beta_BST[:, j], bw_method=0.25)(x), label="BST", color="C5")


    ax_beta.axvline(beta_true_vals[j], alpha=0.9, label="w true", color="black", linestyle="--")
    ax_beta.axvline(beta_GLS[j], alpha=0.9, label="w GLS", color="purple", linestyle=":")
    ax_beta.set_xlabel(fr"$w_{j+1}$", fontsize=15)
    ax_beta.set_ylabel("Density", fontsize=15)
    ax_kappa.set_ylabel("Frequency", fontsize=15)
    #ax_beta.set_title(f"Beta, {titles[j]}")

# One legend for beta panels
handles, labels = axes[1, 0].get_legend_handles_labels()

fig.legend(handles, labels, loc="upper center", ncol=6, bbox_to_anchor=(0.45, 1.0), fontsize=15)
fig.tight_layout(rect=[0, 0, 0.85, 0.93])
fig.savefig("figures_for_use_in_paper/w_vs_kappa.pdf", bbox_inches="tight")
plt.show()
